<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/TurkishNewsArticlesNER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Veri Setini Yükle: TurkishNewsArticles.csv

In [ ]:
import pandas as pd

# TurkishNewsArticles.csv dosyasını yükle
df_news = pd.read_csv('TurkishNewsArticles.csv')

print(df_news.head())

print(df_news.info())

### Tarih Verilerini Dönüştürme



In [ ]:
import locale

try:
    locale.setlocale(locale.LC_ALL, 'tr_TR.UTF-8')
except locale.Error:
    print("Turkish locale not available, falling back to manual month mapping.")
    turkish_month_map = {
        'Ocak': 'January', 'Şubat': 'February', 'Mart': 'March', 'Nisan': 'April',
        'Mayıs': 'May', 'Haziran': 'June', 'Temmuz': 'July', 'Ağustos': 'August',
        'Eylül': 'September', 'Ekim': 'October', 'Kasım': 'November', 'Aralık': 'December'
    }

    def convert_turkish_date(date_str):
        date_str_cleaned = date_str.split(',')[0].strip()
        for tr_month, en_month in turkish_month_map.items():
            date_str_cleaned = date_str_cleaned.replace(tr_month, en_month)
        return date_str_cleaned

    df_news['date_cleaned'] = df_news['date'].apply(convert_turkish_date)
    df_news['date'] = pd.to_datetime(df_news['date_cleaned'], format='%d %B %Y')
    df_news = df_news.drop(columns=['date_cleaned'])


else:
    df_news['date'] = pd.to_datetime(df_news['date'], format='%d %B %Y, %A')



print("DataFrame info after date conversion:")
print(df_news.info())
print("\nFirst few rows with updated date column:")
print(df_news.head())

### Yayın Sıklığı Analizi



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

daily_counts = df_news['date'].value_counts().sort_index()

df_news['week'] = df_news['date'].dt.strftime('%Y-%W')
weekly_counts = df_news['week'].value_counts().sort_index()

df_news['month'] = df_news['date'].dt.to_period('M')
monthly_counts = df_news['month'].value_counts().sort_index()

print("Daily Article Counts:\n", daily_counts.head())
print("\nWeekly Article Counts:\n", weekly_counts.head())
print("\nMonthly Article Counts:\n", monthly_counts.head())

In [ ]:
plt.figure(figsize=(15, 6))
sns.lineplot(x=daily_counts.index, y=daily_counts.values)
plt.title('Daily Article Publication Trends')
plt.xlabel('Date')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 6))
sns.lineplot(x=weekly_counts.index, y=weekly_counts.values)
plt.title('Weekly Article Publication Trends')
plt.xlabel('Week (YYYY-WW)')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 6))
sns.lineplot(x=monthly_counts.index.astype(str), y=monthly_counts.values)
plt.title('Monthly Article Publication Trends')
plt.xlabel('Month (YYYY-MM)')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Yazar Katkısı Analizi



In [ ]:
author_counts = df_news['author'].value_counts()
print("Total articles per author:\n", author_counts.head())

top_10_authors = author_counts.head(10)
print("\nTop 10 Authors by Article Count:\n", top_10_authors)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=top_10_authors.index, y=top_10_authors.values, hue=top_10_authors.index, palette='viridis', legend=False)
plt.title('Top 10 Authors by Article Count')
plt.xlabel('Author')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Metin Uzunluğu Analizi


In [ ]:
df_news['word_count'] = df_news['text'].apply(lambda x: len(str(x).split()))
df_news['char_count'] = df_news['text'].apply(lambda x: len(str(x)))

print("DataFrame with word and character counts:")
print(df_news[['text', 'word_count', 'char_count']].head())

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(df_news['word_count'], bins=50, kde=True)
plt.title('Distribution of Word Counts')
plt.xlabel('Word Count')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(df_news['char_count'], bins=50, kde=True)
plt.title('Distribution of Character Counts')
plt.xlabel('Character Count')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


### Makale Duyarlılık Analizi



In [ ]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[\d]', '', text)
    text = re.sub(r'[\W_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_news['cleaned_text'] = df_news['text'].apply(preprocess_text)

print("Original text vs. Cleaned text (first 5 rows):")
for i in range(5):
    print(f"Original: {df_news['text'].iloc[i][:100]}...")
    print(f"Cleaned:  {df_news['cleaned_text'].iloc[i][:100]}...\n")

### Daha Gelişmiş Türkçe NER Modeli Kullanımı


In [ ]:
import torch
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1

try:
    # Yeni ve daha spesifik NER modeli (akdeniz27/bert-base-turkish-cased-ner denenecek)
    ner_analyzer_specific = pipeline(
        "ner",
        model="akdeniz27/bert-base-turkish-cased-ner", # Model adı tekrar düzeltildi
        aggregation_strategy="simple",
        device=device_id
    )
except Exception as e:
    print(f"Yeni NER modelini yüklerken hata oluştu: {e}")
    ner_analyzer_specific = None

if ner_analyzer_specific:
    texts_list = df_news['cleaned_text'].fillna("").tolist()

    print(f"Yeni NER işlemi { 'A100 GPU' if device_id == 0 else 'CPU' } üzerinde başlatıldı...")

    # Yeni modelle varlıkları çıkar
    ner_results_specific = ner_analyzer_specific(texts_list, batch_size=32)

    df_news['named_entities_specific'] = ner_results_specific
    print("\nYeni model ile işlem başarıyla tamamlandı!")

    # İlk birkaç sonuçtan örnek gösterelim
    for i, entities in enumerate(df_news['named_entities_specific'].head()):
        print(f"\nMakale {i+1} için bulunan varlıklar:")
        for entity in entities:
            print(f"  - Etiket: {entity['entity_group']}, Kelime: {entity['word']}, Skor: {entity['score']:.2f}")
        if i >= 2: # Sadece ilk 3 makaleyi göster
            break
else:
    print("Yeni NER modeli yüklenemediği için işlem yapılamadı.")

### NER Modelinin Varlık Tipi Dağılımını Görselleştirme

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Yeni modelden çıkan varlık tiplerini çıkar
entity_types_specific = defaultdict(int)
if 'named_entities_specific' in df_news.columns:
    for entities_list in df_news['named_entities_specific']:
        if entities_list:
            for entity in entities_list:
                entity_types_specific[entity['entity_group']] += 1

# DataFrame'e dönüştür ve sırala
entity_df_specific = pd.DataFrame(list(entity_types_specific.items()), columns=['Entity Type', 'Count'])
entity_df_specific = entity_df_specific.sort_values(by='Count', ascending=False)

print("\nYeni NER Modeli Varlık Tipi Dağılımı:")
print(entity_df_specific)

# Plot the distribution
plt.figure(figsize=(12, 7))
sns.barplot(x='Entity Type', y='Count', data=entity_df_specific, palette='viridis', hue='Entity Type', legend=False)
plt.title('Yeni NER Modeli Varlık Tipi Dağılımı')
plt.xlabel('Varlık Tipi')
plt.ylabel('Sayım')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
display(entity_df_specific)

print("\n### İlk birkaç makaleden örnek varlıklar:")
for i, entities in enumerate(df_news['named_entities_specific'].head(5)):
    print(f"\nMakale {i+1} için bulunan varlıklar:")
    if entities:
        for entity in entities:
            print(f"  - Etiket: {entity['entity_group']}, Kelime: {entity['word']}, Skor: {entity['score']:.2f}")
    else:
        print("  (Varlık bulunamadı)")

### Adlandırılmış Varlıklar İçin Kelime Bulutu

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

all_named_entity_words = []

if 'named_entities_specific' in df_news.columns:
    for entities_list in df_news['named_entities_specific']:
        if entities_list:
            for entity in entities_list:
                word = entity['word'].replace('##', '')
                if len(word) > 2: #
                    all_named_entity_words.append(word)

# Kelimeleri birleştirerek tek bir metin oluşturalım
text_for_wordcloud = " ".join(all_named_entity_words)

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_for_wordcloud)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('En Çok Geçen Adlandırılmış Varlıklar Kelime Bulutu')
plt.show()